> 按前面几章的方法，系统已经调快了：模型量化了、引擎换了、采样参数也定了。但上线之前还差一个问题没人回答——跑出来的东西还「对」吗？量化降低了精度，换引擎换了 kernel，改一个 temperature 输出就变。每一步都可能让质量悄悄滑落，而你看到的只是 tokens/s 在涨。

> 速度的数字很容易读，质量的数字却不容易读。这一章讲的就是这门手艺：怎么可信地回答「质量有没有掉」，以及「A 是不是真的比 B 好」。我们先拆开一次评测的完整流水线，看哪一环最容易让分数失真；再看出题的 benchmark 和打分的 metric 怎么配合；然后用置信区间回答「差多少才算真的差」；最后亲手跑一条最小评测流水线，做一份上线前的对比清单。


## 1. 评测流水线

很多人把评测想象成「跑个榜」——其实一次评测是一条流水线，每一环都会影响最终那个分数：

```text
Dataset / Benchmark     <- 出什么题
        ↓
Prompt / Chat Template  <- 题目怎么包装给模型
        ↓
Generation config       <- temperature、max_tokens、seed
        ↓
Model / Engine          <- 被测对象
        ↓
Parser                  <- 从输出里抠出答案
        ↓
Metric or Judge         <- 怎么打分
        ↓
Aggregation + 置信区间   <- 怎么汇总
```

「模型 A 比模型 B 高 1 分」要成立，前提是这条流水线上下两次运行**除了被测对象，一环都不变**。换一个 prompt 模板、换一个 parser、甚至只换了 temperature，分数的变动可能比模型本身的差距还大。后面几节会逐环看最容易出事的地方。

## 2. 评测对象的分类

「评测」这个词至少装着四种不同的问题，混在一起谈会乱：

- **Model quality**：模型本身的能力——知识、数学、代码、指令遵循
- **推理质量回归**：量化 / 换 kernel / 换引擎之后，输出有没有变差——Part 3 最关心的
- **Serving 性能**：TTFT、TPOT、吞吐、显存——上一本的指标
- **系统成本**：每百万 Token 的钱、GPU 数、功耗

厂商报告常把 quality 和 performance 放在同一张图上，因为工程优化的本质是 trade-off。看一个典型例子：

In [ ]:
configs = {
    "BF16": {"memory_gb":14.0, "quality":72.4, "throughput":1.0},
    "INT8": {"memory_gb":7.2,  "quality":72.2, "throughput":1.35},
    "INT4": {"memory_gb":3.8,  "quality":71.3, "throughput":1.75},
}
for name,v in configs.items():
    print(name, v)
print("真正的问题不是 INT4 快不快，而是：省下的显存/吞吐，值不值得质量下降。")


三行数字放在一起，问题就不再是「INT4 快不快」，而是「省下的显存和吞吐，值不值得那 1.1 分的质量下降」。这个判断没有标准答案，但做出它至少需要两样东西：可信的质量分数（本章），和可信的性能数字（上一本）。先看质量这半边怎么测。

## 3. Benchmark 与 Metric

两个词经常被混用，其实是流水线上两环。**Benchmark** 定义题目：MMLU 是上万道四选一的选择题，GSM8K 是几千道小学数学应用题，HumanEval 是 164 个编程题。**Metric** 定义怎么得分：accuracy 数对了几道，exact match 要求答案逐字相同，pass@k 允许生成 k 次只要有一次通过。

常见搭配和它们测的东西：

| Benchmark | 测什么 | 常用 Metric |
|:---|:---|:---|
| MMLU / GPQA | 知识与推理 | accuracy |
| GSM8K / MATH | 数学 | exact match / 解析后比对 |
| HumanEval / LiveCodeBench | 代码 | pass@k（真的运行测试） |
| SWE-bench | 真实仓库级修 bug | 测试是否通过 |
| 开放式对话 | 主观质量 | pairwise preference / LLM-as-Judge |

比背清单更重要的是那个问题意识：**这个榜单测的能力，和你关心的能力是一回事吗？** GSM8K 满分不代表会写周报。同一个 benchmark 换 prompt、换 few-shot 数量、换 parser，分数也会变——所以读报告时先翻附录看它的完整流水线配置。

### 常见 Benchmark 的题目长什么样

光看名字记不住它们测什么。这一节把最常出现在报告里的几个 benchmark 各掏出一道**真实风格的例题**，你看一眼题目本身，就明白它考的能力是什么、答案怎么判。

先给四个词的正式定义：

- **MMLU**（Massive Multitask Language Understanding）：一道英文选择题题库，覆盖 57 个学科，从高中物理到法学，用 accuracy 判分。
- **C-Eval**：中国团队构建的中文选择题 benchmark，覆盖 52 个学科，从中小学到执业资格，四选一。
- **CMMLU**：另一个中文选择题 benchmark，题材更偏中文语境常识（成语、地理、传统医学等）。
- **GSM8K**：小学难度的英文数学应用题，答案是一个数字，用 exact match 或解析后比对判分。

下面用代码把每类题的「题干 + 选项 + 标准答案 + 判分方式」打印出来（例题按原题风格改写，真实题库在各自的数据集里）：

In [ ]:
benchmarks = {
    "MMLU（英文四选一）": {
        "subject": "高中物理",
        "question": "A 2 kg object accelerates at 3 m/s^2. What is the net force?",
        "choices": ["1.5 N", "6 N", "5 N", "0.67 N"],
        "answer": "B",
        "metric": "accuracy（比对选项字母）",
    },
    "C-Eval（中文四选一）": {
        "subject": "高中地理",
        "question": "我国地势的基本特征是？",
        "choices": ["东高西低", "西高东低，呈阶梯状分布", "南高北低", "四周高、中间低"],
        "answer": "B",
        "metric": "accuracy（比对选项字母）",
    },
    "CMMLU（中文语境常识）": {
        "subject": "初中语文",
        "question": "成语「亡羊补牢」的意思最接近下面哪一项？",
        "choices": ["为时已晚，无济于事", "出了问题后想办法补救，可以防止继续受损失",
                    "形容做事非常谨慎", "比喻虚有其表"],
        "answer": "B",
        "metric": "accuracy（比对选项字母）",
    },
    "GSM8K（数学应用题）": {
        "subject": "小学数学",
        "question": "Natalia sold clips to 48 friends in April, and half as many in May. "
                    "How many clips did she sell altogether?",
        "choices": None,
        "answer": "72",
        "metric": "exact match（解析出最终数字比对）",
    },
}

for name, item in benchmarks.items():
    print(f"=== {name}（{item['subject']}）")
    print("题干:", item["question"])
    if item["choices"]:
        for letter, c in zip("ABCD", item["choices"]):
            print(f"  {letter}. {c}")
    print("标准答案:", item["answer"], " 判分:", item["metric"])
    print()


看题时有三个值得注意的细节：

1. **选择题不生成，只挑选项**。MMLU / C-Eval / CMMLU 的标准玩法是把 A/B/C/D 四个选项的概率算出来取最大的（loglikelihood 判分），比让模型自由生成再解析稳定得多。你看到的很多榜单分数，模型根本没「说话」，只是在选。
2. **中文题不等于英文题的翻译**。C-Eval 和 CMMLU 是独立出题的，考「执业药师」「中医基础」这类中文语境的学科，一个英文满分模型在中文榜上可能掉很多分——所以中文业务必须看中文榜。
3. **数学题的答案藏在解题过程后面**。GSM8K 的原始数据在答案前有一段逐步推理，判分时只取最后的数字。这正好呼应第 1 节流水线里的 Parser 一环：解析失败 ≠ 做错，很多「丢分」其实是解析问题。

代码类 benchmark（HumanEval / LiveCodeBench）长得很不一样：给一个函数签名和 docstring，要求补全实现，然后**真的运行**单元测试来判断通过与否（pass@k）。它的判分不是比对文本，而是执行代码——这也是为什么代码榜的分数可信度高：测试通过就是通过，没有模糊空间。

## 4. LLM-as-Judge 的偏差

选择题可以数对错，开放式输出（摘要、对话、文案）怎么打分？人工最准但最贵，于是流行用强模型当裁判：把两个模型的输出并排给裁判，让它选谁更好。

方便，但裁判自己带一堆已知偏差：position bias（偏向某个固定位置）、length bias（偏爱长回答）、style bias（偏爱格式漂亮的）、self-preference（偏爱和自己像的）。对策也简单直接：固定评分标准（rubric）、**双序重判**（A/B 交换位置各判一次，两次结论不一致就记为平局或人工仲裁）、保留裁判的原始输出以便审计。

「交换位置重判」能查出多少问题？用一个带已知偏差的模拟裁判试一次：

In [ ]:
import random

def biased_judge(golden_position, bias=0.15):
    """模拟 judge：85% 按真实质量判，15% 无条件偏向位置 1"""
    if random.random() < bias:
        return 1
    return golden_position

random.seed(42)
pairs = 200
flips = 0
for _ in range(pairs):
    golden = random.choice([1, 2])          # 更好的答案在哪边是随机的
    picked1 = biased_judge(golden)          # 第一次判：位置就是答案编号
    picked2 = 3 - biased_judge(3 - golden)  # 交换后：位置 1 变成答案 2，换算回来
    if picked1 != picked2:
        flips += 1

print(f"{pairs} 对答案，交换顺序重判后 {flips} 对结论翻转（{flips / pairs:.0%}）")
print()
print("关键观察：judge 只有 15% 的判决偏向位置 1，就有 15% 的对比结论被翻转——")
print("偏差不会被平均掉，每一份偏差都直接变成不可信的结论；双序重判能把它暴露出来")

In [ ]:
# 一致 vs 翻转的对比：judge 的结论并没有想象中稳定
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 3))
plt.bar(["consistent", "flipped"], [pairs - flips, flips],
        color=["tab:green", "tab:red"])
plt.ylabel("pairs")
plt.title(f"Position bias flips {flips / pairs:.0%} of verdicts")
plt.show()

结果值得咂摸：裁判只有 15% 的判决偏向位置 1，最终就有 15% 的对比结论被翻转——偏差不会被「平均」掉，每一份偏差都直接变成不可信的结论。双序重判就是把这些翻转暴露出来的最低成本手段。

## 5. 置信区间

还有一类误差来自采样本身。20 道题对 15 道（75%）和对 14 道（70%），谁更强？直觉上 75% 更好——但 20 道题的样本太小，一次重跑的运气成分完全可能大于这 5 分。

处理这类不确定性的标准工具是 bootstrap 置信区间：把已有的答卷有放回地重抽很多次，每次算一个分数，看这些分数的分布范围。20 对 15 的例子：

In [ ]:
import random, statistics

random.seed(42)
scores = [1]*15 + [0]*5

boots=[]
for _ in range(5000):
    sample=[random.choice(scores) for _ in scores]
    boots.append(sum(sample)/len(sample))

boots.sort()
print("accuracy:", sum(scores)/len(scores))
print("95% bootstrap interval:", round(boots[125],3), round(boots[-126],3))


区间 [0.55, 0.90] 意味着：仅凭这 20 道题，真实水平可能低到 55%，也可能高到 90%。两个模型的置信区间只要重叠，那几分差距就不足以宣布胜负——要么加题量，要么承认「测不出来」。

把这条原则记牢，再看模型榜单上的 76.1 vs 76.4，你的第一反应应该是：区间是多少？

## 6. 最小评测流水线实战

不依赖任何评测框架，把本章第 1 节那条流水线亲手走一遍：出题 → 让「模型」作答 → 解析答案 → 算分 → 置信区间。被测模型用 mock：以可调的概率答对，而且输出格式随机变化——正好把 parser 这一环的坑暴露出来。

真实场景里，把 `mock_model` 换成对 OpenAI-compatible 端点的调用，其余环节一行不用改——这正是流水线思维的价值：换模型不换流程。

In [ ]:
import random
import re

random.seed(42)

# 1) Dataset：12 道有唯一数字答案的小题（教学用例，真实评测会用标准 benchmark）
questions = [
    ("小明有 3 个苹果，又买了 5 个，一共几个？", 8),
    ("一辆车每小时行 60 公里，2 小时行多少公里？", 120),
    ("一打鸡蛋有 12 个，两打有几个？", 24),
    ("一件衣服原价 100 元，打 8 折是多少钱？", 80),
    ("一个班有 30 人，一半是女生，女生有几人？", 15),
    ("从 1 加到 10 等于多少？", 55),
    ("一根绳子长 9 米，剪掉 4 米还剩几米？", 5),
    ("电影 7 点开始，长 2 小时，几点结束？", 9),
    ("一本书 240 页，每天读 80 页，几天读完？", 3),
    ("一箱水 24 瓶，喝掉 10 瓶还剩几瓶？", 14),
    ("3 乘 7 等于多少？", 21),
    ("温度从 5 度上升 8 度，现在是几度？", 13),
]

# 2) 被「评测」的 mock 模型：正确率可调，输出格式随机变化
def mock_model(question, answer, correctness=0.7):
    """以 correctness 的概率答对；错时给一个相近但错误的数"""
    correct = random.random() < correctness
    value = answer if correct else answer + random.choice([1, 2, 3])
    style = random.choice(["plain", "cn", "en"])
    if style == "plain":
        return str(value)
    if style == "cn":
        return f"答案是 {value}。"
    return f"The answer is: {value}."

# 3) Parser：从模型输出里抽取最后一个整数
def parse_answer(text):
    numbers = re.findall(r"\d+", text)
    return int(numbers[-1]) if numbers else None

for text in [mock_model(questions[0][0], questions[0][1]) for _ in range(3)]:
    print(f"输出: {text!r:<28} -> 解析: {parse_answer(text)}")
print("关键观察：同一个模型三种输出格式，parser 必须全都接得住")

mock 的三种输出格式（纯数字、中文句式、英文句式）就是真实模型的写照——同一个模型对不同问题的回答格式五花八门，parser 必须全都接得住。接下来跑两个「正确率不同」的模型，用本章第 5 节的 bootstrap 给分数配上区间：

In [ ]:
# 4) 跑两个「模型」，算 accuracy 和 bootstrap 置信区间
def run_eval(model_fn):
    results = []
    for q, gold in questions:
        pred = parse_answer(model_fn(q, gold))
        results.append(1 if pred == gold else 0)
    return results

def bootstrap_ci(results, n_boot=5000):
    stats = []
    for _ in range(n_boot):
        sample = random.choices(results, k=len(results))
        stats.append(sum(sample) / len(sample))
    stats.sort()
    return stats[int(0.025 * n_boot)], stats[int(0.975 * n_boot)]

random.seed(11)
scores_a = run_eval(lambda q, a: mock_model(q, a, correctness=0.75))
scores_b = run_eval(lambda q, a: mock_model(q, a, correctness=0.60))

acc_a, acc_b = sum(scores_a) / len(scores_a), sum(scores_b) / len(scores_b)
lo_a, hi_a = bootstrap_ci(scores_a)
lo_b, hi_b = bootstrap_ci(scores_b)

print(f"模型 A: accuracy {acc_a:.2f}  95% CI [{lo_a:.2f}, {hi_a:.2f}]")
print(f"模型 B: accuracy {acc_b:.2f}  95% CI [{lo_b:.2f}, {hi_b:.2f}]")
print()
print("关键观察：A 看起来更高，但两个置信区间重叠——")
print("12 道题的规模下，这个差距不足以证明 A 真的更强")

In [ ]:
# 带误差棒的对比图：区间重叠 = 差距不显著
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.2))
for i, (name, acc, lo, hi) in enumerate([
        ("model A", acc_a, lo_a, hi_a), ("model B", acc_b, lo_b, hi_b)]):
    plt.errorbar(i, acc, yerr=[[acc - lo], [hi - acc]], fmt="o", capsize=6)
plt.xticks([0, 1], ["model A", "model B"])
plt.ylabel("accuracy (with 95% CI)")
plt.ylim(0, 1.05)
plt.title("Overlapping CIs: the gap is not conclusive at n=12")
plt.show()

模型 A 看着领先 17 分，但两个置信区间大面积重叠——12 道题根本不够分辨这个量级的差距。这个结论本身就是评测的产出：「测不出来」比一个误导性的 0.75 vs 0.58 有用得多。

### 评测库：把流水线交给现成工具

第 1 节说过，一次评测是七环流水线。自己管七环很容易出错，社区早就有了标准工具，先认三个最常用的：

- **lm-evaluation-harness**（EleutherAI）：事实上的英文评测标准工具。内置几百个 task（mmlu、gsm8k、hellaswag……），把第 1 节流水线里「出题、包装 prompt、判分、汇总」全部接管，你只需要给模型和 task 名。
- **OpenCompass**（上海 AI 实验室）：国内最常用的评测框架，中英 benchmark 都全，C-Eval / CMMLU / GSM8K 一条命令跑，司南大榜就是它跑的。
- **EvalScope**（阿里）：魔搭社区（ModelScope）配套的评测框架，和 vLLM 等 inference 后端集成好，侧重「模型上线前的评测 + 压测」一条龙，中文支持好。

用 lm-evaluation-harness 跑一次 MMLU 大致长这样（示意，本节不用真的执行）：

```bash
lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-1.5B \
    --tasks mmlu \
    --batch_size 8
```

换一个评测对象只需换 `--tasks`：`ceval`、`cmmlu`、`gsm8k`、`hellaswag` 都是内置 task 名。用 OpenCompass 则是写一份 config 指定「模型 × 数据集 × 策略」，它负责跑完汇总成表。

选工具时仍然用第 1 节的框架问自己：**它替我固定了哪几环，我还得自己管哪几环？** 例如 harness 固定了题库和判分，但 chat template 怎么包、generation config 是什么，仍是你自己的责任——换了个模板，分数就不可比了。此外还有几类定位不同的工具：

In [ ]:
tools = [
    ("lm-evaluation-harness", "EleutherAI", "英文标准榜最通用，task 最全，学术报告标配"),
    ("OpenCompass", "上海AI实验室", "中英都全，司南榜背后引擎，国内模型报告常用"),
    ("EvalScope", "阿里 ModelScope", "与 vLLM/MaaS 集成好，评测+压测一条龙，中文友好"),
    ("lighteval", "Hugging Face", "与 transformers/datasets 生态结合紧密"),
    ("AlpacaEval / MT-Bench", "学术界", "开放式对话质量，LLM-as-Judge 判分（前面 LLM-as-Judge 一节讲过它的偏差问题）"),
    ("Promptfoo / DeepEval", "应用层", "写进 CI 的回归测试，自定义断言"),
]
print(f"{'工具':<28}{'来自':<14}定位")
print("-" * 80)
for name, org, use in tools:
    print(f"{name:<30}{org:<14}{use}")
print()
print("关键观察：前四个是「跑标准榜」的工具，后两个是「守自己业务」的工具，")
print("两者互补——榜单分数防选型失误，业务回归测试防上线事故。")


## 7. 评测库与工具地图

具体介绍放在本章第 3 节「评测库」小节。这里只补两类容易混淆的东西：

- **SWE-bench harness**：代码任务连仓库带测试一起跑，跑的不是 prompt 而是真实工程环境
- **vllm bench serve / SGLang bench**：上一本的 serving 性能压测，测的是速度不是质量

工具会一直更新，流水线不会。选工具的依据始终是「它替我固定了哪几环、我还得自己管哪几环」。

## 8. 上线前的最小对比

把本章收拢成一份可直接照做的清单。以「要不要从 BF16 换到 AWQ INT4」为例，固定的部分：

```text
model revision / tokenizer / chat template
dataset（题目集合）
generation config（temperature=0 或固定 seed）
max context / max output
hardware / concurrency
```

同时记录的部分：

```text
Quality: benchmark 分数 + 置信区间
Memory : 峰值显存
TTFT   : P50 / P95
TPOT   : P50 / P95
Throughput: tokens/s
```

固定与记录齐了，才可能回答那个真正的问题：INT4 省下的显存和吞吐，值不值得这份质量差异。下一章就带着这份清单，把模型真正部署成服务：

## 小结

- 一次评测是流水线：题目、模板、生成配置、模型、parser、指标、汇总——除被测对象外一环都不能变
- Benchmark 出题，Metric 打分；先问「这个榜测的能力和我要的一致吗」
- parser 是最常被忽略、最容易翻车的一环
- LLM-as-Judge 有 position / length / style 偏差——双序重判是最低成本的体检
- 分数必须配置信区间；区间重叠的差距不足为凭
- 质量分数（本章）+ 性能数字（上一本）+ 固定的 workload = 可做的上线决策

最后一章完成闭环：

> **把 checkpoint 交给 vLLM / SGLang，暴露成 API，亲手测出自己系统的 TTFT / TPOT / 吞吐。**

## 作业

三道题覆盖 pipeline 的三个易翻车点：置信区间、parser、judge 偏差。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。

### 作业 1：实现 bootstrap 置信区间

有放回地重采样、每次算均值、排序后取 2.5% 和 97.5% 两个分位点。

**小提示**：`random.choices(results, k=len(results))` 一步完成有放回抽样。

In [ ]:
# 作业 1：bootstrap 置信区间 填空

import random

def bootstrap_ci(results, n_boot=2000):
    """返回 (下界, 上界)"""
    stats = []
    for _ in range(n_boot):
        # TODO：把下面三引号里的内容替换成你的代码
        """有放回抽 len(results) 个，把均值放进 stats"""
    stats.sort()
    return stats[int(0.025 * n_boot)], stats[int(0.975 * n_boot)]

random.seed(0)
lo, hi = bootstrap_ci([1] * 9 + [0] * 3)   # 12 题对 9 题
assert lo <= 0.75 <= hi
assert lo < 0.55 and hi > 0.9              # 区间足够宽：小样本就是不自信
print("✅ 作业 1 通过：分数必须配上区间才有意义")

### 作业 2：写一个更稳的答案 parser

真实输出里的数字可能带千分位逗号或单位：`1,234`、`42 tokens`。parser 要剥掉逗号，
并取最后一个数字。

**小提示**：先 `text.replace(",", "")` 去掉逗号，再 `re.findall(r"\d+", ...)` 取最后一个。

In [ ]:
# 作业 2：parser 增强版 填空

import re

def parse_answer(text):
    """从输出里抽最后一个整数（剥掉千分位逗号）；没有数字返回 None"""
    # TODO：把下面三引号里的内容替换成你的代码
    """先去掉逗号，再找出所有整数取最后一个转 int；找不到返回 None"""

assert parse_answer("The answer is: 1,234.") == 1234
assert parse_answer("used 42 tokens, cost 3 tokens") == 3
assert parse_answer("no number here") is None
print("✅ 作业 2 通过：parser 是评测 pipeline 里最容易翻车的一环")

### 作业 3：量化 position bias

双序各判一次，两次选中的「答案」不一致就叫翻转；翻转率就是 position bias 的直接证据。

**小提示**：第二次判完记得把位置换算回答案编号（`3 - 位置`）。

In [ ]:
# 作业 3：翻转率 填空

import random

def flip_rate(judge, pairs=100):
    """judge(golden_position) 返回 1 或 2；返回双序重判的结论翻转率"""
    flips = 0
    for _ in range(pairs):
        golden = random.choice([1, 2])
        picked1 = judge(golden)
        picked2 = 3 - judge(3 - golden)   # 交换顺序后换算回答案编号
        # TODO：把下面三引号里的内容替换成你的代码
        """picked1 与 picked2 不一致时 flips 加一"""
    return flips / pairs

random.seed(0)
rate = flip_rate(lambda g: 1 if random.random() < 0.3 else g)
assert 0.2 < rate < 0.65, rate
print("✅ 作业 3 通过：你会用双序重判给 judge 做体检了")